# Generate part04 data for 2021 JLab Hackathon

This notebook is used to generate the data sets for part04 of the 2021 Hackathon.

This is nearly identical to the part01 problem except the number of showers generated is randomly sampled from a poisson distribution. Having zero showers is allowed. The only label is the number of showers in the event. One potential gotcha is that there is no mechansim to prevent showers from overlapping. In the case where two showers are very close, it would be essentially impossible to tell that it was not a single shower.

Three data sets are produced: train, test, and judge. The first two are provided for the participant to train on. The "judge" dataset is in two forms. The one provided to the participants has the labels removed. The form with the labels intact are stored in the "restricted" directory.

All data sets come in two forms. One that is pure csv where the first 900 values represent the calorimeter energy deposition information followed by the labels. The second has the first element as the name of a PNG file followed by the labels. This gives participants the option of which format they prefer to use. In all cases, the events are identical so they may use the pure csv format to train, but can still look at the pre-made PNG image for a particular event for visualization.

This first section defines several gobal parameters defining how to create the data set.

In [1]:
import math

#------------------------------------------
# Configuration
Nblocks  = 30
Nevents_train   = 10000
Nevents_test    =  2000
Nevents_judge   =  2000
NclustersMin    =     1
NclustersMax    =     3

Xmin            = -1000.0
Xmax            = +1000.0
ClusterSize     = 1.0  # cluster std dev in blockwidths
ClusterAmpMean  = 3.0  # Mean amplitude of cluster 
ClusterAmpSigma = 2.0  # Standard deviation of cluster amplitude
ClusterAmpMin   = 0.5  # Minimum cluster amplitude
ClusterAmpMax   = 8.0  # Maximum cluster amplitude
ClusterPosSigma = 8.0  # Standard deviation of Gaussian used to sample position of cluster

#------------------------------------------
# Calculated values
binwidth = (Xmax-Xmin)/Nblocks
one_over_sqrt2_sigma = 1.0 / (math.sqrt(2) * ClusterSize*binwidth)
int_gauss = 2.0*math.erf(binwidth/2.0*one_over_sqrt2_sigma)
max_integral = ClusterAmpMax*int_gauss*int_gauss  # maximum value a block can have


The following is just a utility to print an updating progress bar so we can monitor the longer running cells.

In [2]:
#------------------------------------------
# Code for displaying progess bar (ripped from internet: https://www.mikulskibartosz.name/how-to-display-a-progress-bar-in-jupyter-notebook/)
import time, sys
from IPython.display import clear_output

def update_progress(label, progress):
    bar_length = 20
    if isinstance(progress, int):
        progress = float(progress)
    if not isinstance(progress, float):
        progress = 0
    if progress < 0:
        progress = 0
    if progress >= 1:
        progress = 1

    block = int(round(bar_length * progress))

    clear_output(wait = True)
    text = label + ": [{0}] {1:.1f}%".format( "#" * block + "-" * (bar_length - block), progress * 100)
    print(text)

## Procedures to create single events and full event sets in CSV format

This is where the events are generated and the block values defined. The routine
is used to create training, test, and judging data sets. 
n.b. The judging sets will have the labels separated manually in a later cell. 

Here, we create "pure" events. A cluster is defined by a point randomly selected
to be in the detector. The centroid may be at the border, but not beyond it.
The distribution in x/y is sampled from a Gaussian that is centered on the center
of the detector (0,0) with a sigma that is ClusterPosSigma blocks. This is done to
give the position a non-uniform distribution, but still wide enough to ensure
events cover the entire detector.

The amplitude of the cluster is randomly selected, also from a Gaussian distribution,
but with limits defined by ClusterAmpMin and ClusterAmpMax. 

Each cell of the NblocksxNblocks array is set equal to the integral of a 2D
Gaussian centered on the cluster, but with a sigma of ClusterSize\*binwidth. The
Gaussian itself is not properly normalized, but this is not really needed since the
"energy" units of the clusters are arbitrary.

It turns out that the integral of a 2D Guassian over a square block is proportional
to the product of 2 erf differences. We only need to calculate the erf(x2)-erf(x1)
for each column(row) and the cell is the product of these.

In [3]:
import numpy as np
import math
import os

#------------------------------------------
# MakeEvent
#------------------------------------------
def MakeEvent():

    Nclusters = np.random.randint(NclustersMin, NclustersMax+1)
    
    Eblock = [0.0] * Nblocks * Nblocks
    labels = []
    for icluster in range(Nclusters):
        
        # Sample amplitude of cluster
        amp = 0.0
        while amp<ClusterAmpMin or amp>ClusterAmpMax : amp = np.random.normal( ClusterAmpMean, ClusterAmpSigma )
        
        # Sample X/Y coordinate of cluster
        x0 = -10000.0
        y0 = -10000.0
        while x0<Xmin or x0>Xmax : x0 = np.random.normal( 0.0, ClusterPosSigma*binwidth )
        while y0<Xmin or y0>Xmax : y0 = np.random.normal( 0.0, ClusterPosSigma*binwidth )
        
        # Calculate factors for integrals over blocks
        xfac = [0]*(Nblocks+1)
        yfac = [0]*(Nblocks+1)
        for i in range(Nblocks+1):
            x = Xmin + i*binwidth
            a0 = (x-x0)*one_over_sqrt2_sigma
            a1 = (x+binwidth-x0)*one_over_sqrt2_sigma
            xfac[i] = math.erf( a1 ) - math.erf( a0 )
        for i in range(Nblocks+1):
            y = Xmin + i*binwidth
            a0 = (y-y0)*one_over_sqrt2_sigma
            a1 = (y+binwidth-y0)*one_over_sqrt2_sigma
            yfac[i] = math.erf( a1 ) - math.erf( a0 )
       
        # Set all block values
        for i in range(Nblocks):
            for j in range(Nblocks):
                idx = i + Nblocks*j
                Eblock[idx] += (amp*xfac[i]*yfac[j])
        
        # Add to labels
        labels += [amp,x0,y0]

    # Add no-cluster values to labels
    while len(labels) < 3*(NclustersMax-NclustersMin+1):
        labels += [-1,-1,-1]
    
    # Create CSV output string for this event
    event = ','.join(['%6.4f' % x for x in Eblock]) + ',' + ','.join(['%6.4f' % x for x in labels])
        
    #print('amp:%f  max(Eblock): %f  max(xfac): %f   max(yfac): %f  ' % (amp, max([float(x) for x in Eblock]), max(xfac), max(yfac)))

    return event

#------------------------------------------
# MakeDataSet
#------------------------------------------
def MakeDataSet(directory, setname, Nevents):

    # Create directory (if needed) and open CSV file
    os.makedirs(directory, exist_ok=True)
    of = open( os.path.join(directory, setname + '.csv'), 'w')
    label = 'Generating ' + os.path.join(directory, setname)

    for ievent in range(Nevents):
        event_csv = MakeEvent() # Make one event w/ labels
        of.write( event_csv + '\n')
        if ievent%100 == 0 : update_progress(label, ievent / Nevents) # Periodically update progress bar
    of.close() # Close output CSV file    
    update_progress(label, 1)
    print('Done.')

## Generate all part04 data sets

In [4]:
MakeDataSet('part04', 'train', Nevents_train)

Generating part04/train: [####################] 100.0%
Done.


In [5]:
MakeDataSet('part04', 'test' , Nevents_test)

Generating part04/test: [####################] 100.0%
Done.


In [6]:
MakeDataSet('part04', 'judge', Nevents_judge)

Generating part04/judge: [####################] 100.0%
Done.


## Generate image files from CSV

Here a CSV file is read and a PNG is produced for each event and written to a dedicated images directory. A new CSV file is also created in the output directory which replaces the Nblocks x Nblocks raw image information with just the image file name while maintaining the labels. This is to give participants an option to use images instead of pure CSV data. It also provides them with an easy visual aid to any specific event.

In [7]:
import csv
from PIL import Image
import matplotlib.pyplot as plt
import os

def GenerateImages(directory, setname):

    image_dirname = os.path.join(directory, setname+'_images')
    os.makedirs(image_dirname, exist_ok=True)
    label = 'Generating PNG images ' + os.path.join(directory, setname)

    with open( os.path.join(directory, setname + '.csv'), newline='' ) as csvfile:
        
        Nevents = sum(1 for line in csvfile)
        csvfile.seek(0) # rewind
        
        newcsv_file = open( os.path.join(directory, setname + '_images.csv'), 'w' )
      
        reader = csv.reader(csvfile)
        for i, line in enumerate(reader):

            # Periodically update progress bar
            if i%100 == 0 : update_progress(label, i / Nevents)

            X = np.array([255.0*float(x)/max_integral for x in line[:Nblocks*Nblocks]]).reshape(Nblocks,Nblocks).astype(np.uint8)
            img = Image.fromarray(X)
            fname = os.path.join(image_dirname, 'event%06d.png' % i)
            img.save(fname)

            rel_fname = '/'.join(fname.split('/')[1:]) # drop first directory from path (i.e. 'part01/') so CSV only contains setname/filename.png
            newcsv_line = ', '.join([rel_fname]+line[Nblocks*Nblocks:])
            newcsv_file.write(newcsv_line + '\n')
 
        newcsv_file.close()

    # Final update of progress bar
    update_progress(label, 1)
    print('Done.')

Matplotlib created a temporary config/cache directory at /tmp/matplotlib-0pmi9e9w because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [8]:
 GenerateImages('part04', 'train')

Generating PNG images part04/train: [####################] 100.0%
Done.


In [9]:
 GenerateImages('part04', 'test')

Generating PNG images part04/test: [####################] 100.0%
Done.


In [10]:
GenerateImages('part04', 'judge')

Generating PNG images part04/judge: [####################] 100.0%
Done.


## Display some images

In [1]:
from IPython.display import HTML

Npics_per_row = 4
pic_width="300px"

mess = '<h3>NOTE: Images may appear correctly due to scaling algorithm of browser!</h3><br>'

#---------------------
# Train
mess += '<table><tr>'
mess += '<td colspan="100"> <center>TRAIN</center> </td>\n'
mess += '</tr><tr>\n'
for i in range(Npics_per_row):
    fname = 'part04/train_images/event%06d.png' % i
    mess += '<td><img width="' + pic_width + '" src="' + fname + '"><br><font size="-1">' + fname + '</td>\n'
mess += '</tr></table>'
display(HTML(mess))

#---------------------
# Test
mess = '<table><tr>'
mess += '<td colspan="100"> <center>TEST</center> </td>\n'
mess += '</tr><tr>\n'
for i in range(Npics_per_row):
    fname = 'part04/test_images/event%06d.png' % i
    mess += '<td><img width="' + pic_width + '" src="' + fname + '"><br><font size="-1">' + fname + '</td>\n'
mess += '</tr></table>'
display(HTML(mess))

#---------------------
# Judge
mess = '<table><tr>'
mess += '<td colspan="100"> <center>JUDGE</center> </td>\n'
mess += '</tr><tr>\n'
for i in range(Npics_per_row):
    fname = 'part04/judge_images/event%06d.png' % i
    mess += '<td><img width="' + pic_width + '" src="' + fname + '"><br><font size="-1">' + fname + '</td>\n'
mess += '</tr></table>'
display(HTML(mess))


## Scrub Judging Files

In this cell the csv files for the "judge" data set are moved to the directory "restricted" and replaced with versions that have the label information removed.

A check is made so we don't accidentally lose all of the labels if the cell is run multiple times.

In [12]:
import os
import shutil
import csv

# Because this replaces a file with one of the same name but containing
# less information, there is the possibility the label information would
# be lost if we ran this cell twice in a row. Thus, we check here if the
# source has the labels and raise an exception if it doesn't
if not os.path.exists('part04/judge.csv'):
    raise ValueError('File part04/judge.csv does not exist!')
with open( 'part04/judge.csv', newline='' ) as tmpin:
    reader = csv.reader(tmpin)
    if len(reader.__next__()) <= (Nblocks*Nblocks):
        raise ValueError('File part04/judge.csv already has labels removed!')
    tmpin.close()


# Make sure "restricted" directory exists and move files with labels there
os.makedirs('restricted', exist_ok=True)
if os.path.exists('part04/judge.csv'):
    shutil.move('part04/judge.csv', 'restricted/part04_judge.csv')
if os.path.exists('part04/judge_images.csv'):
    shutil.move('part04/judge_images.csv', 'restricted/part04_judge_images.csv')

#---------------------------
# part03/judge.csv
with open( 'restricted/part04_judge.csv', newline='' ) as csvin:  
        with open( 'part04/judge.csv', 'w', newline='' ) as csvout:
      
            # Get number of events
            Nevents = sum(1 for line in csvin)
            csvin.seek(0) # rewind
        
            reader = csv.reader(csvin)
            writer = csv.writer(csvout)
            for i, line in enumerate(reader):
                writer.writerow( line[:Nblocks*Nblocks] )  # write only feature info
            csvin.close()
            csvout.close()
                
#---------------------------
# part03/judge_images.csv
with open( 'restricted/part04_judge_images.csv', newline='' ) as csvin:  
        with open( 'part04/judge_images.csv', 'w', newline='' ) as csvout:
      
            # Get number of events
            Nevents = sum(1 for line in csvin)
            csvin.seek(0) # rewind
        
            reader = csv.reader(csvin)
            writer = csv.writer(csvout)
            for i, line in enumerate(reader):
                writer.writerow( line[:1] )  # write only feature info
            csvin.close()
            csvout.close()
             